In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif, chi2
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Chargement des données
df = pd.read_csv('data_clean_recon.csv')
X = df.drop('label', axis=1)
y = df['label']

# --- ÉTAPE 1 : EXTRACTION DE STATISTIQUES (Feature Engineering) ---
stats_df = df.groupby('device_name_enc')['message_val_num'].agg(['mean', 'std', 'var']).fillna(0)
stats_df.columns = ['val_mean', 'val_std', 'val_var']

# Fusion avec le dataset principal
df = df.join(stats_df, on='device_name_enc')
df['is_high_traffic'] = (df['message_val_num'] > df['message_val_num'].mean()).astype(int)

# --- SAUVEGARDE 1 : data_enriched.csv ---
df.to_csv('data_enriched.csv', index=False)

# Mise à jour de X après enrichissement
X = df.drop('label', axis=1)

# --- ÉTAPE 2 : SÉLECTION DE FEATURES (ANOVA & Chi2) ---
selector_anova = SelectKBest(score_func=f_classif, k=4)
X_anova = selector_anova.fit_transform(X, y)
cols_anova = X.columns[selector_anova.get_support()]

selector_chi2 = SelectKBest(score_func=chi2, k=4)
X_chi2 = selector_chi2.fit_transform(X, y)
cols_chi2 = X.columns[selector_chi2.get_support()]

# --- ÉTAPE 3 : RÉDUCTION DE DIMENSION (PCA) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2) 
X_pca = pca.fit_transform(X_scaled)

print("Caractéristiques sélectionnées par ANOVA:", list(cols_anova))
print("Caractéristiques sélectionnées par Chi2:", list(cols_chi2))
print(f"Variance expliquée par les 2 composantes PCA: {pca.explained_variance_ratio_.sum():.2%}")

# --- ÉTAPE 4 : PRÉPARATION & SMOTE ---
selected_features = ['message_type_enc', 'message_val_num', 'val_mean', 
                    'is_high_traffic', 'application_enc', 'ip_enc', 'topic_enc']
X = df[selected_features]
y = df['label']

# Division Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Application de SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# --- SAUVEGARDE 2 & 3 : train_smote_ready.csv et test_ready.csv ---

# Création du fichier Train (équilibré)
train_smote_ready = pd.DataFrame(X_train_res, columns=selected_features)
train_smote_ready['label'] = y_train_res
train_smote_ready.to_csv('train_smote_ready.csv', index=False)

# Création du fichier Test (original)
test_ready = X_test.copy()
test_ready['label'] = y_test
test_ready.to_csv('test_ready.csv', index=False)

print("-" * 30)
print("Fichiers générés avec succès :")
print("1. data_enriched.csv (Complet + Stats)")
print("2. train_smote_ready.csv (Features sélectionnées + SMOTE)")
print("3. test_ready.csv (Features sélectionnées, sans SMOTE)")

Caractéristiques sélectionnées par ANOVA: ['message_type_enc', 'message_val_num', 'val_mean', 'is_high_traffic']
Caractéristiques sélectionnées par Chi2: ['application_enc', 'ip_enc', 'topic_enc', 'message_val_num']
Variance expliquée par les 2 composantes PCA: 71.16%
------------------------------
Fichiers générés avec succès :
1. data_enriched.csv (Complet + Stats)
2. train_smote_ready.csv (Features sélectionnées + SMOTE)
3. test_ready.csv (Features sélectionnées, sans SMOTE)
